# Task 5.1 — Build the Before/After Comparison Table

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, accuracy_score, f1_score, roc_auc_score
import joblib

df = pd.read_excel('Practice_Dataset.xlsx')

# --- Regression setup ---
features = ['punch_count', 'hours_worked', 'satisfaction_score']
df_model = df[features + ['monthly_salary']].copy()
imputer = SimpleImputer(strategy='median')
df_model[features + ['monthly_salary']] = imputer.fit_transform(df_model[features + ['monthly_salary']])
X = df_model[features]
y = df_model['monthly_salary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Classification setup (with Day 3/4 fix: fillna instead of dropna) ---
clf_features = ['punch_count', 'hours_worked', 'satisfaction_score']
df_clf = df[clf_features + ['is_absent']].copy()
df_clf['hours_worked'] = df_clf['hours_worked'].fillna(0)
df_clf['satisfaction_score'] = df_clf['satisfaction_score'].fillna(df_clf['satisfaction_score'].median())
Xc = df_clf[clf_features]
yc = df_clf['is_absent']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=42, stratify=yc
)

In [4]:
# Train all 4 regression stages from Days 1-3 to fill the table with real values
lin = LinearRegression().fit(X_train, y_train)
rf_unc = RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42).fit(X_train, y_train)
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=5, random_state=42).fit(X_train, y_train)

param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [4, 8, 12, None], 'min_samples_leaf': [1, 5, 10]}
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

comparison = pd.DataFrame([
    {'Model': 'Linear Regression (baseline)', 'Stage': 'Before',
     'Train R2': round(r2_score(y_train, lin.predict(X_train)), 3),
     'Test R2': round(r2_score(y_test, lin.predict(X_test)), 3)},
    {'Model': 'Random Forest (unconstrained)', 'Stage': 'Before',
     'Train R2': round(r2_score(y_train, rf_unc.predict(X_train)), 3),
     'Test R2': round(r2_score(y_test, rf_unc.predict(X_test)), 3)},
    {'Model': 'Random Forest (regularized)', 'Stage': 'After',
     'Train R2': round(r2_score(y_train, rf_reg.predict(X_train)), 3),
     'Test R2': round(r2_score(y_test, rf_reg.predict(X_test)), 3)},
    {'Model': 'Random Forest (GridSearchCV tuned)', 'Stage': 'After',
     'Train R2': round(r2_score(y_train, grid_search.predict(X_train)), 3),
     'Test R2': round(r2_score(y_test, grid_search.predict(X_test)), 3)},
])
comparison['Gap'] = comparison['Train R2'] - comparison['Test R2']
comparison.to_csv('model_comparison_before_after.csv', index=False)
print(comparison)

                                Model   Stage  Train R2  Test R2    Gap
0        Linear Regression (baseline)  Before     0.014    0.042 -0.028
1       Random Forest (unconstrained)  Before     0.490   -1.168  1.658
2         Random Forest (regularized)   After     0.158   -0.017  0.175
3  Random Forest (GridSearchCV tuned)   After     0.058    0.026  0.032


# Task 5.2 — Classification Before/After

In [6]:
default_clf = RandomForestClassifier(random_state=42).fit(Xc_train, yc_train)
def_pred = default_clf.predict(Xc_test)
def_proba = default_clf.predict_proba(Xc_test)[:, 1]

param_grid_clf = {'n_estimators': [50, 100, 200], 'max_depth': [4, 8, None], 'class_weight': [None, 'balanced']}
grid_clf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_clf, cv=5, scoring='f1', n_jobs=-1)
grid_clf.fit(Xc_train, yc_train)
best_clf = grid_clf.best_estimator_
tuned_pred = best_clf.predict(Xc_test)
tuned_proba = best_clf.predict_proba(Xc_test)[:, 1]

clf_comparison = pd.DataFrame([
    {'Model': 'Random Forest (default)',
     'Accuracy': round(accuracy_score(yc_test, def_pred), 3),
     'F1': round(f1_score(yc_test, def_pred), 3),
     'AUC': round(roc_auc_score(yc_test, def_proba), 3)},
    {'Model': 'Random Forest (GridSearchCV tuned)',
     'Accuracy': round(accuracy_score(yc_test, tuned_pred), 3),
     'F1': round(f1_score(yc_test, tuned_pred), 3),
     'AUC': round(roc_auc_score(yc_test, tuned_proba), 3)},
])
clf_comparison.to_csv('classification_comparison_before_after.csv', index=False)
print(clf_comparison)

                                Model  Accuracy   F1  AUC
0             Random Forest (default)       1.0  1.0  1.0
1  Random Forest (GridSearchCV tuned)       1.0  1.0  1.0


# Task 5.3 — Save the Final Tuned Models

In [9]:
import joblib

# Save the best regression model from GridSearchCV/RandomizedSearchCV
joblib.dump(grid_search.best_estimator_, 'best_regression_model.pkl')

# Save the best classifier
joblib.dump(grid_clf.best_estimator_, 'best_classifier_model.pkl')

print('Saved best_regression_model.pkl and best_classifier_model.pkl')

Saved best_regression_model.pkl and best_classifier_model.pkl


# Task 5.4 — Write the Results Report


# RESULTS REPORT --- Week 5: Model Evaluation & Hyperparameter Optimization

==========================================================================

Analyst: Fajer Alshammari

REGRESSION TASK (monthly_salary):
- Diagnosis: underfit observed in Linear Regression (train R2=0.014, val R2=-0.12)
  via learning curve; overfit observed in unconstrained Random Forest
  (train R2=0.49, val R2=-1.19, gap=1.67) via learning curve.
- Fix applied: regularization (max_depth, min_samples_leaf) + GridSearchCV tuning
- Before: Train R2 = 0.490, Test R2 = -1.168, Gap = 1.658 (RF unconstrained)
- After:  Train R2 = 0.058, Test R2 = 0.026,  Gap = 0.032 (RF GridSearchCV tuned)
- Best hyperparameters: {'max_depth': 4, 'min_samples_leaf': 10, 'n_estimators': 100}

CLASSIFICATION TASK (is_absent):
- Confusion matrix insight: no errors of either type in this run (TN=65, FP=0,
  FN=0, TP=7) -- driven by punch_count perfectly separating absent vs present,
  a data-leakage concern flagged in Days 3-4, not genuine model skill.
- AUC before tuning: 1.000, AUC after tuning: 1.000
- Best hyperparameters: {'class_weight': None, 'max_depth': 4, 'n_estimators': 50}

KEY TAKEAWAY:
Regularization and systematic tuning shrank the regression train/test gap from
1.658 to 0.032, turning an overfit model into one that generalizes (test R2 went
from negative to positive) -- but overall predictive power stayed weak (test R2
~0.03), meaning the three chosen features explain very little of monthly_salary
and better features matter more here than further tuning.

FILES PRODUCED:
- model_comparison_before_after.csv
- classification_comparison_before_after.csv
- best_regression_model.pkl, best_classifier_model.pkl
- learning curve, ROC, and precision-recall plots
